# 第 02 章：从脆弱字符串到可验证业务契约（概念实验与工程迁移）

按正文顺序完成每个实验：先写预测，再运行代码，阅读输出，最后修改一个变量。

概念实验不会预先导入 Mini DeerFlow；进入“工程迁移”标签后，才把同一机制放回项目。

## 实验 1：改一个标签就让计划解析器崩溃

`concept` · `failure` · `research-request`

**运行前先预测**：模型把“目标”改写成“研究主题”后，依赖固定标签的解析器会怎样？

> 先在这里写下你的判断，再执行下一个代码单元。

In [1]:
def parse_labeled_plan(text: str) -> dict[str, object]:
    fields = {}
    for line in text.splitlines():
        key, value = line.split("：", 1)
        fields[key] = value
    return {
        "objective": fields["目标"],
        "steps": [item.strip() for item in fields["步骤"].split("→")],
    }


model_text = (
    "研究主题：解释 LangGraph durable execution\n"
    "步骤：检索官方资料 → 整理机制 → 生成报告"
)
print("model_text =", model_text.replace("\n", " | "))
try:
    parse_labeled_plan(model_text)
except KeyError as error:
    assert error.args == ("目标",)
    print("KeyError: required label '目标' is missing")
else:
    raise AssertionError("固定标签变化后应暴露解析失败")


model_text = 研究主题：解释 LangGraph durable execution | 步骤：检索官方资料 → 整理机制 → 生成报告
KeyError: required label '目标' is missing


**发生了什么**：回答语义没有变化，字符串协议却已断裂。继续增加正则别名，只会把模型语言的全部可能变化塞进解析器。
真正的设计问题是：下游需要哪些字段、类型和约束？这些要求应成为显式 Schema，而不是藏在 Prompt 与 split 代码里。

**动手修改**：让步骤分隔符从 `→` 变成编号列表。记录需要继续增加多少分支，直到你愿意停止修补字符串格式。

## 实验 2：定义最小 ResearchRequest 并验证范围

`concept` · `repair` · `research-request`

**运行前先预测**：`max_sources="4"` 会被转换成整数吗？值为 0 时会在哪里失败？

> 先在这里写下你的判断，再执行下一个代码单元。

In [2]:
from pydantic import BaseModel, Field, ValidationError


class ResearchRequest(BaseModel):
    question: str = Field(min_length=1)
    deliverable: str = Field(min_length=1)
    max_sources: int = Field(ge=1, le=8)


request_candidate = {
    "question": "LangGraph 如何恢复长任务？",
    "deliverable": "带引用的中文说明",
    "max_sources": "4",
}
validated_request = ResearchRequest.model_validate(request_candidate)
print("request =", validated_request.model_dump())
print("max_sources_type =", type(validated_request.max_sources).__name__)

try:
    ResearchRequest.model_validate({**request_candidate, "max_sources": 0})
except ValidationError as error:
    first_error = error.errors()[0]
    print("invalid_field =", first_error["loc"])
    print("error_type =", first_error["type"])
else:
    raise AssertionError("来源预算必须大于零")


request = {'question': 'LangGraph 如何恢复长任务？', 'deliverable': '带引用的中文说明', 'max_sources': 4}
max_sources_type = int
invalid_field = ('max_sources',)
error_type = greater_than_equal


**发生了什么**：候选字符串被确定性转换成整数，越界值在业务代码执行前被拒绝。调用方可以依赖字段名和错误位置，而不是解析异常文本。
Schema 合法只证明结构满足约束，不证明问题真实、来源可信或计划质量足够。这些属于后续检索与评测。

**动手修改**：把 `max_sources` 改成无法转换的字符串，再比较错误类型。然后决定严格模式是否更符合你的 API 边界。

## 实验 3：用 `with_structured_output` 得到 ResearchRequest

`concept` · `baseline` · `model-structured-output`

**运行前先预测**：模型返回的 AIMessage 正文为空时，结构化调用结果仍能成为 Pydantic 对象吗？

> 先在这里写下你的判断，再执行下一个代码单元。

In [3]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from pydantic import BaseModel, Field


class ModelResearchRequest(BaseModel):
    question: str = Field(min_length=1)
    deliverable: str = Field(min_length=1)
    max_sources: int = Field(ge=1, le=8)


class StructuredFakeModel(GenericFakeChatModel):
    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        del tools, tool_choice, kwargs
        return self


raw_structured_model = StructuredFakeModel(
    messages=iter(
        [
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "ModelResearchRequest",
                        "args": {
                            "question": "LangGraph 如何恢复长任务？",
                            "deliverable": "带引用的中文说明",
                            "max_sources": 4,
                        },
                        "id": "structured-request-1",
                        "type": "tool_call",
                    }
                ],
            )
        ]
    )
)
structured_model = raw_structured_model.with_structured_output(
    ModelResearchRequest
)
model_request = structured_model.invoke("把用户需求整理成研究请求")

print("result_type =", type(model_request).__name__)
print("question =", model_request.question)
print("max_sources =", model_request.max_sources)


result_type = ModelResearchRequest
question = LangGraph 如何恢复长任务？
max_sources = 4


**发生了什么**：模型层先产生符合 Schema 的 tool call，LangChain 再解析并验证为 Pydantic 对象。这一步没有执行业务工具，也没有创建 Agent 循环。
Provider-native JSON Schema 与 tool strategy 只是生成候选结构的不同方式。无论使用哪种策略，业务仍要执行自己的 Pydantic 和领域校验。

**动手修改**：让 tool call 缺少 `deliverable`。观察异常发生在结构化解析边界，而不是后续 Graph 节点。

## 实验 4：让缺失目标静默变成“继续处理”

`concept` · `failure` · `task-plan`

**运行前先预测**：payload 只有 steps 时，Pydantic 会拒绝，还是生成一个看似完整的计划？

> 先在这里写下你的判断，再执行下一个代码单元。

In [4]:
from pydantic import BaseModel, Field


class UnsafePlan(BaseModel):
    objective: str = "继续处理"
    steps: list[str] = Field(default_factory=list)


unsafe_plan = UnsafePlan.model_validate(
    {"steps": ["检索官方资料", "生成报告"]}
)
print("plan_is_valid =", isinstance(unsafe_plan, UnsafePlan))
print("objective =", unsafe_plan.objective)
print("objective_was_in_payload =", "objective" in unsafe_plan.model_fields_set)
print("steps =", unsafe_plan.steps)


plan_is_valid = True
objective = 继续处理
objective_was_in_payload = False
steps = ['检索官方资料', '生成报告']


**发生了什么**：Schema 合法，但目标并非来自模型或用户。错误被推迟到检索、写文件甚至发布阶段，调用方还难以区分真实目标与默认补值。

**动手修改**：给 `steps` 也设置一个看似合理的默认步骤。列出后续系统会在哪些位置误以为计划已经确认。

## 实验 5：让目标必填并验证步骤依赖

`concept` · `repair` · `task-plan`

**运行前先预测**：依赖不存在的步骤 ID 时，错误应落在执行 Graph，还是计划进入系统之前？

> 先在这里写下你的判断，再执行下一个代码单元。

In [5]:
from pydantic import BaseModel, Field, ValidationError, model_validator


class PlanStep(BaseModel):
    id: str = Field(min_length=1)
    instruction: str = Field(min_length=1)
    depends_on: list[str] = Field(default_factory=list)


class TaskPlan(BaseModel):
    schema_version: int = 1
    objective: str = Field(min_length=1)
    steps: list[PlanStep] = Field(min_length=1)

    @model_validator(mode="after")
    def dependencies_exist(self):
        ids = {step.id for step in self.steps}
        missing = sorted(
            dependency
            for step in self.steps
            for dependency in step.depends_on
            if dependency not in ids
        )
        if missing:
            raise ValueError(f"unknown dependencies: {missing}")
        return self


task_plan = TaskPlan(
    objective="解释 LangGraph durable execution",
    steps=[
        PlanStep(id="research", instruction="检索官方资料"),
        PlanStep(
            id="write",
            instruction="生成带引用的中文说明",
            depends_on=["research"],
        ),
    ],
)
print("plan =", task_plan.model_dump())

try:
    TaskPlan(
        objective="错误计划",
        steps=[PlanStep(id="write", instruction="写报告", depends_on=["missing"])],
    )
except ValidationError as error:
    print("dependency_error =", error.errors()[0]["type"])
else:
    raise AssertionError("未知依赖必须在计划边界被拒绝")


plan = {'schema_version': 1, 'objective': '解释 LangGraph durable execution', 'steps': [{'id': 'research', 'instruction': '检索官方资料', 'depends_on': []}, {'id': 'write', 'instruction': '生成带引用的中文说明', 'depends_on': ['research']}]}
dependency_error = value_error


**发生了什么**：目标和步骤不再由默认值伪造；依赖引用也在计划进入执行层之前完成校验。`schema_version` 为 checkpoint、数据集和 API 的历史 payload 提供迁移入口。

**动手修改**：加入重复步骤 ID。先预测当前 validator 是否会发现，再补充唯一性规则，并说明循环依赖还需要什么检查。

## 实验 6：让普通字符串路径逃出工作区

`concept` · `failure` · `artifact-contract`

**运行前先预测**：只声明 `path: str` 时，Pydantic 会拒绝 `../secret.txt` 吗？

> 先在这里写下你的判断，再执行下一个代码单元。

In [6]:
from pydantic import BaseModel


class UnsafeArtifact(BaseModel):
    path: str
    media_type: str


unsafe_artifact = UnsafeArtifact(
    path="../secret.txt",
    media_type="text/plain",
)
print("artifact_valid =", isinstance(unsafe_artifact, UnsafeArtifact))
print("accepted_path =", unsafe_artifact.path)
print("contains_parent_segment =", ".." in unsafe_artifact.path.split("/"))


artifact_valid = True
accepted_path = ../secret.txt
contains_parent_segment = True


**发生了什么**：`str` 只约束 Python 类型，没有表达“工作区内相对路径”的领域规则。文件工具若直接使用该值，可能越过预期目录。

**动手修改**：再尝试绝对路径和空路径。整理 Artifact path 的最小确定性规则，但不要声称它已经解决符号链接和容器隔离。

## 实验 7：用字段 validator 拒绝绝对路径和父目录

`concept` · `repair` · `artifact-contract`

**运行前先预测**：合法相对路径是否原样保留？非法路径的错误位置会指向哪个字段？

> 先在这里写下你的判断，再执行下一个代码单元。

In [7]:
from pathlib import PurePosixPath

from pydantic import BaseModel, Field, ValidationError, field_validator


class ArtifactRef(BaseModel):
    path: str = Field(min_length=1)
    media_type: str = Field(min_length=1)

    @field_validator("path")
    @classmethod
    def workspace_relative_path(cls, value: str) -> str:
        path = PurePosixPath(value)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError("artifact path 必须是工作区内的相对路径")
        return value


valid_artifact = ArtifactRef(
    path="reports/answer.md",
    media_type="text/markdown",
)
print("valid_artifact =", valid_artifact.model_dump())

try:
    ArtifactRef(path="../secret.txt", media_type="text/plain")
except ValidationError as error:
    path_error = error.errors()[0]
    print("invalid_field =", path_error["loc"])
    print("error_type =", path_error["type"])
else:
    raise AssertionError("父目录路径必须被拒绝")


valid_artifact = {'path': 'reports/answer.md', 'media_type': 'text/markdown'}
invalid_field = ('path',)
error_type = value_error


**发生了什么**：Schema 现在表达工作区相对路径规则，错误也稳定定位到 `path`。这仍不是完整 Sandbox。
符号链接解析、真实文件根目录、原子写入、provider 生命周期和 shell 隔离会在 Sandbox 专题通过实际文件系统失败继续推导。

**动手修改**：测试 `reports//answer.md`、`.` 和 Windows 风格路径。决定是否规范化，并说明规范化必须发生在校验前还是之后。

## 实验 8：用三种互不相同的方式表达结果

`concept` · `failure` · `structured-failure`

**运行前先预测**：调用方要用多少种分支才能处理成功、拒答和无效字段？

> 先在这里写下你的判断，再执行下一个代码单元。

In [8]:
from pydantic import BaseModel, Field, ValidationError


class OutcomeRequest(BaseModel):
    question: str = Field(min_length=1)
    max_sources: int = Field(ge=1)


def inconsistent_parse(payload: dict[str, object]):
    if payload.get("refusal"):
        return None
    return OutcomeRequest.model_validate(payload)


success_value = inconsistent_parse(
    {"question": "解释 checkpoint", "max_sources": 3}
)
refusal_value = inconsistent_parse({"refusal": "未授权数据"})
try:
    inconsistent_parse({"question": "", "max_sources": 0})
except ValidationError:
    validation_channel = "exception"
else:
    validation_channel = "return-value"

print("success_channel =", type(success_value).__name__)
print("refusal_channel =", type(refusal_value).__name__)
print("validation_channel =", validation_channel)
print("caller_protocols =", 3)


success_channel = OutcomeRequest
refusal_channel = NoneType
validation_channel = exception
caller_protocols = 3


**发生了什么**：三类结果使用对象、`None` 和异常三条通道。Graph、API 和评测器若各自处理，分支很快产生漂移。

**动手修改**：再加入超时字符串 `"timeout"`。统计调用方需要增加多少类型判断，并思考哪些异常仍应上抛。

## 实验 9：用显式失败对象统一拒答与校验错误

`concept` · `repair` · `structured-failure`

**运行前先预测**：返回类型固定为 Request 或 StructuredFailure 后，调用方还能否区分拒答与字段错误？

> 先在这里写下你的判断，再执行下一个代码单元。

In [9]:
from typing import Literal

from pydantic import BaseModel, Field, ValidationError


class StableRequest(BaseModel):
    question: str = Field(min_length=1)
    max_sources: int = Field(ge=1)


class StructuredFailure(BaseModel):
    kind: Literal["refusal", "validation_error"]
    message: str
    fields: list[str] = Field(default_factory=list)


def stable_parse(payload: dict[str, object]) -> StableRequest | StructuredFailure:
    if reason := payload.get("refusal"):
        return StructuredFailure(kind="refusal", message=str(reason))
    try:
        return StableRequest.model_validate(payload)
    except ValidationError as error:
        fields = [str(item["loc"][0]) for item in error.errors()]
        return StructuredFailure(
            kind="validation_error",
            message="请求字段无效",
            fields=fields,
        )


stable_success = stable_parse({"question": "解释 checkpoint", "max_sources": 3})
stable_refusal = stable_parse({"refusal": "未授权数据"})
stable_invalid = stable_parse({"question": "", "max_sources": 0})

print("success_type =", type(stable_success).__name__)
print("refusal =", stable_refusal.model_dump())
print("validation =", stable_invalid.model_dump())


success_type = StableRequest
refusal = {'kind': 'refusal', 'message': '未授权数据', 'fields': []}
validation = {'kind': 'validation_error', 'message': '请求字段无效', 'fields': ['question', 'max_sources']}


**发生了什么**：调用方只需处理成功或失败对象，失败的 `kind` 仍能穷尽分支。普通程序 bug、取消和系统错误不应被伪装成 StructuredFailure。

**动手修改**：加入 `timeout` kind，并决定 retryable 是否属于失败对象。写出 API、Graph 和评测器各自消费哪些字段。

## 实验 10：对照模型输出、工具参数和 Agent 最终响应

`concept` · `contrast` · `schema-lifetimes`

**运行前先预测**：研究计划、检索 query 和最终报告元数据应该使用同一个 Schema 吗？

> 先在这里写下你的判断，再执行下一个代码单元。

In [10]:
schema_lifetimes = [
    {
        "boundary": "model_output",
        "example": "ResearchRequest / TaskPlan",
        "validated": "一次模型回答解析时",
    },
    {
        "boundary": "tool_args",
        "example": "SearchQuery",
        "validated": "工具执行之前",
    },
    {
        "boundary": "agent_response",
        "example": "ReportMetadata",
        "validated": "完整工具循环结束时",
    },
]

for item in schema_lifetimes:
    print(
        f"{item['boundary']}: {item['example']} -> {item['validated']}"
    )


model_output: ResearchRequest / TaskPlan -> 一次模型回答解析时
tool_args: SearchQuery -> 工具执行之前
agent_response: ReportMetadata -> 完整工具循环结束时


**发生了什么**：研究计划属于模型输出契约；检索 query 属于工具参数契约；最终报告元数据属于 Agent 完整循环的响应契约。
把三者混用，会让错误在错误层级重试。第 04 章才会实现工具参数与 Agent 循环；本章只把差异记录为后续接口约束。

**动手修改**：为“用户上传文件路径”选择边界。说明它为什么还需要 Sandbox 规则，而不能只依赖工具 args Schema。

## 实验 11：运行项目 Schema 并观察稳定序列化结果

`migration` · `contrast` · `research-request`

**运行前先预测**：TaskPlan 是否包含 schema version？非法 Artifact 是否仍在项目边界被拒绝？

> 先在这里写下你的判断，再执行下一个代码单元。

In [11]:
from pydantic import ValidationError

from mini_deerflow.schemas import (
    ArtifactRef,
    PlanStep,
    ResearchRequest,
    StructuredFailure,
    TaskPlan,
    validate_research_request,
)


project_request = ResearchRequest(
    question="LangGraph 如何恢复长任务？",
    deliverable="带引用的中文说明",
    max_sources=4,
)
project_plan = TaskPlan(
    objective="解释 LangGraph durable execution",
    steps=[
        PlanStep(id="research", instruction="检索官方资料"),
        PlanStep(id="write", instruction="写报告", depends_on=["research"]),
    ],
)
project_invalid = validate_research_request(
    {"question": "", "deliverable": "报告", "max_sources": 0}
)
try:
    ArtifactRef(path="../secret.txt", media_type="text/plain")
except ValidationError:
    artifact_rejected = True
else:
    artifact_rejected = False

print("request =", project_request.model_dump())
print("plan_schema_version =", project_plan.schema_version)
print("plan_dependencies =", project_plan.steps[1].depends_on)
print("invalid_kind =", project_invalid.kind)
print("artifact_rejected =", artifact_rejected)
print("failure_type =", isinstance(project_invalid, StructuredFailure))


request = {'question': 'LangGraph 如何恢复长任务？', 'deliverable': '带引用的中文说明', 'max_sources': 4}
plan_schema_version = 1
plan_dependencies = ['research']
invalid_kind = validation_error
artifact_rejected = True
failure_type = True


**发生了什么**：Mini DeerFlow 将请求、计划、Artifact 和失败类型作为公共协议复用，避免教程、Graph、API 与测试各自定义一份近似 Schema。
`SubagentResult` 不在本章首次教学。它会在第 11 章先经历委派失败，再迁移到项目的 Subagent 输出协议。